In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import signal
from scipy.io import loadmat
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from scipy.signal import medfilt
from torch.utils.data import WeightedRandomSampler
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
def normalise(data, train_reps):
    """
    Normalise function rewritten to support different numbers of channels
    """
    x = [np.where(data.values[:, data.shape[1] - 2] == rep) for rep in train_reps]
    indices = np.squeeze(np.concatenate(x, axis=-1))
    train_data = data.iloc[indices, :]
    train_data = train_data.reset_index(drop=True)

    scaler = StandardScaler(with_mean=True,
                            with_std=True,
                            copy=False).fit(train_data.iloc[:, :data.shape[1] - 2])

    scaled = scaler.transform(data.iloc[:, :data.shape[1] - 2])
    normalised = pd.DataFrame(scaled)
    normalised['stimulus'] = data['stimulus']
    normalised['repetition'] = data['repetition']
    return normalised


def filter_data(data, f, butterworth_order=4, btype='lowpass'):
    """
    Filter Data function rewritten to support different numbers of channels
    """
    emg_data = data.values[:, :data.shape[1] - 2]

    f_sampling = 2000
    nyquist = f_sampling / 2
    if isinstance(f, int):
        fc = f / nyquist
    else:
        fc = np.array(f, dtype=float) / nyquist
    b, a = signal.butter(butterworth_order, fc, btype=btype)
    transpose = emg_data.T.copy()
    for i in range(len(transpose)):
        transpose[i] = (signal.lfilter(b, a, transpose[i]))
    filtered = pd.DataFrame(transpose.T)
    filtered['stimulus'] = data['stimulus']
    filtered['repetition'] = data['repetition']
    return filtered

def windowing(data, reps, gestures, win_len, win_stride):
    """
    Windowing function rewritten to support different numbers of channels
    """
    if reps:
        x = [np.where(data.values[:, data.shape[1] - 1] == rep) for rep in reps]
        indices = np.squeeze(np.concatenate(x, axis=-1))
        data = data.iloc[indices, :]
        data = data.reset_index(drop=True)
    if gestures:
        x = [np.where(data.values[:, data.shape[1] - 2] == move) for move in gestures]
        indices = np.squeeze(np.concatenate(x, axis=-1))
        data = data.iloc[indices, :]
        data = data.reset_index(drop=True)
    
    idx = [i for i in range(win_len, len(data), win_stride)]
    X = np.zeros([len(idx), win_len, len(data.columns) - 2])
    y = np.zeros([len(idx), ])
    reps = np.zeros([len(idx), ])

    for i, end in enumerate(idx):
        start = end - win_len
        X[i] = data.iloc[start:end, 0:data.shape[1] - 2].values
        mid = start + win_len // 2
        y[i] = data.iloc[mid, data.shape[1] - 2]

        reps[i] = data.iloc[end, data.shape[1] - 1]

    return X, y, reps


def notch_filter(data, f0, Q, fs=2000):
    """
    Notch Filter function rewritten to support different numbers of channels
    """
    emg_data = data.values[:, :data.shape[1] - 2]

    b, a = signal.iirnotch(f0, Q, fs)
    transpose = emg_data.T.copy()

    for i in range(len(transpose)):
        transpose[i] = (signal.lfilter(b, a, transpose[i]))

    filtered = pd.DataFrame(transpose.T)
    filtered['stimulus'] = data['stimulus'].values
    filtered['repetition'] = data['repetition'].values

    return filtered

def load_data(path):
    mat = loadmat(path) 
    emg = mat['emg']

    print(f"subject #{mat['subject']}") # subject number
    print(f"exercise #{mat['exercise']}") # exercise number

    data = pd.DataFrame(mat['emg'])
    data['stimulus'] = mat['restimulus']
    data['repetition'] = mat['repetition']
    return data

def preprocessing(path):
    data = load_data(path)
    return preprocessing_internals(data)

def preprocessing_internals(data):
    emg_low = filter_data(data=data, f=150, butterworth_order=4, btype='lowpass')

    emg_notch = notch_filter(data=emg_low,f0=60,Q=30,fs=2000)

    gestures = data['stimulus'].unique().tolist() 
    train_reps, test_reps = train_test_split(
        data['repetition'].unique().tolist(),
        test_size=0.3,
        random_state=42,
        shuffle=True
    )

    emg_norm = normalise(data = emg_notch, train_reps=train_reps)

    win_len = 200    
    win_stride = 50

    X_train, y_train, r_train = windowing(emg_norm, train_reps, gestures, win_len, win_stride)
    X_test, y_test, r_test = windowing(emg_norm, test_reps, gestures, win_len, win_stride)
    overlap = np.intersect1d(r_train, r_test)
    print(pd.Series(y_train).value_counts())
    print(pd.Series(y_test).value_counts())

    if len(overlap) != 0:
        raise Exception(f'repetitions leaked between train and test for repetition {overlap}')
    
    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights_dict = dict(zip(np.unique(y_train), class_weights))

    return X_train, y_train, X_test, y_test, class_weights_dict

# path = "../NinaProData"

def multi_preprocess(exercise_number, path):
    num_subjects = 27
    x_train_list, y_train_list, x_test_list, y_test_list = [], [], [],[]
    for i in range(num_subjects):
        base_path = f"{path}/s{i + 1}/S{i + 1}_A1_E{exercise_number}.mat"
        data = load_data(base_path)
        x_train, y_train, x_test, y_test, _cw = preprocessing_internals(data)
        x_train_list.append(x_train)
        y_train_list.append(y_train)
        x_test_list.append(x_test)
        y_test_list.append(y_test)

    x_train = np.concatenate(x_train_list)
    y_train = np.concatenate(y_train_list)
    x_test = np.concatenate(x_test_list)
    y_test = np.concatenate(y_test_list)

    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights_dict = dict(zip(np.unique(y_train), class_weights))
    x_test = np.transpose(x_test, (0, 2, 1))
    x_train = np.transpose(x_train, (0, 2, 1))
    
    return x_train, y_train, x_test, y_test, class_weights_dict
class NinaProDataset(torch.utils.data.Dataset):
  def __init__(self, x, y):
    self.X = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)
  
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [53]:
class GLU(nn.Module):
    def __init__(self, in_size):
        super().__init__()
        self.linear1 = nn.Linear(in_size, in_size)
        self.linear2 = nn.Linear(in_size, in_size)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # x: (batch, time, channels) or (batch, seq_len, features)
        a = F.relu(self.linear1(x))
        b = torch.sigmoid(self.linear2(x))
        x = a * b
        return self.dropout(x)

In [54]:
class ConvGLUBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=5, padding=2):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=padding)
        self.bn = nn.BatchNorm1d(out_ch)
        # gating branch uses 1x1 conv on input
        self.gate = nn.Conv1d(in_ch, out_ch, kernel_size=1)
        self.res_proj = None
        if in_ch != out_ch:
            self.res_proj = nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x):
        # x: (batch, channels, time)
        y = self.conv(x)
        y = self.bn(y)
        g = torch.sigmoid(self.gate(x))
        out = F.relu(y * g)
        res = x if self.res_proj is None else self.res_proj(x)
        return F.relu(out + res)

In [55]:
X_train, y_train, X_test, y_test, class_weights = preprocessing('s1/S1_A1_E1.mat')
glu = GLU(10)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
output = glu(X_train_tensor)
print("GLU output shape:", output.shape)

subject #[[1]]
exercise #[[1]]
0.0     353
5.0      58
3.0      53
1.0      50
8.0      44
6.0      43
4.0      41
12.0     40
10.0     39
2.0      37
7.0      36
9.0      35
11.0     35
Name: count, dtype: int64
0.0     910
3.0      30
1.0      26
10.0     25
5.0      22
12.0     21
4.0      18
6.0      17
7.0      17
9.0      17
2.0      16
8.0      16
11.0     15
Name: count, dtype: int64
GLU output shape: torch.Size([864, 202, 10])


In [56]:
class ResGLUClassifier(nn.Module):
    def __init__(self, in_channels, num_classes, base_channels=32):
        super().__init__()
        self.stem = nn.Conv1d(in_channels, base_channels, kernel_size=7, padding=3)
        self.block1 = ConvGLUBlock(base_channels, base_channels * 2, kernel=5, padding=2)
        self.block2 = ConvGLUBlock(base_channels * 2, base_channels * 4, kernel=5, padding=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(base_channels * 4, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        # x: (batch, time, channels)
        x = x.permute(0, 2, 1)            # -> (batch, channels, time)
        x = F.relu(self.stem(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool(x).squeeze(-1)     # (batch, channels)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


In [57]:
def compute_accuracy(model, dataloader, device='cpu'):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


In [58]:
#X_train = (X_train - X_train.mean()) / X_train.std()
#X_test  = (X_test - X_train.mean()) / X_train.std()

In [59]:
train_dataset = NinaProDataset(X_train, y_train)
test_dataset = NinaProDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [60]:
def effective_class_weights(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    weights = (1 - beta) / (1 - beta ** counts)
    weights = weights / weights.sum() * len(classes)
    return torch.tensor(weights, dtype=torch.float32)


In [61]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

num_classes = len(np.unique(y_train))
model = ResGLUClassifier(in_channels=10, num_classes=num_classes).to(device)
classes = np.unique(y_train)
#weights = compute_class_weight("balanced", classes=classes, y=y_train)
#weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max', factor=0.5, patience=15)

num_epochs = 200

for epoch in range(num_epochs):
    model.train()
    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    # Accuracy assessment
    train_acc = compute_accuracy(model, train_loader, device)
    test_acc = compute_accuracy(model, test_loader, device)

    scheduler.step(test_acc)

    print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")


Epoch 1: Loss=2.2229, Train Acc=0.5243, Test Acc=0.8235
Epoch 2: Loss=1.6738, Train Acc=0.5637, Test Acc=0.8409
Epoch 3: Loss=1.2508, Train Acc=0.5880, Test Acc=0.8557
Epoch 4: Loss=0.9416, Train Acc=0.7326, Test Acc=0.8826
Epoch 5: Loss=0.8487, Train Acc=0.8137, Test Acc=0.8974
Epoch 6: Loss=0.4137, Train Acc=0.8449, Test Acc=0.9070
Epoch 7: Loss=0.7452, Train Acc=0.8831, Test Acc=0.9235
Epoch 8: Loss=0.4918, Train Acc=0.8715, Test Acc=0.9087
Epoch 9: Loss=0.4983, Train Acc=0.9248, Test Acc=0.9357
Epoch 10: Loss=0.3436, Train Acc=0.9097, Test Acc=0.9278
Epoch 11: Loss=0.3752, Train Acc=0.7558, Test Acc=0.6043
Epoch 12: Loss=0.2311, Train Acc=0.9514, Test Acc=0.9296
Epoch 13: Loss=0.2111, Train Acc=0.9259, Test Acc=0.9426
Epoch 14: Loss=0.3097, Train Acc=0.9340, Test Acc=0.9313
Epoch 15: Loss=0.2508, Train Acc=0.9213, Test Acc=0.9452
Epoch 16: Loss=0.4024, Train Acc=0.9306, Test Acc=0.9322
Epoch 17: Loss=0.1156, Train Acc=0.9630, Test Acc=0.9383
Epoch 18: Loss=0.0874, Train Acc=0.8785,

In [62]:

from sklearn.metrics import classification_report
def get_all_predictions(model, loader, device):
    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(y.numpy())
    y_pred = medfilt(np.array(y_pred), kernel_size=5)
    return y_true, y_pred

In [63]:
y_true, y_pred = get_all_predictions(model, test_loader, device)
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       910
           1       0.77      0.88      0.82        26
           2       0.94      0.94      0.94        16
           3       0.94      0.57      0.71        30
           4       0.94      0.89      0.91        18
           5       0.96      1.00      0.98        22
           6       1.00      0.94      0.97        17
           7       0.94      0.94      0.94        17
           8       0.73      1.00      0.84        16
           9       1.00      0.94      0.97        17
          10       0.65      0.68      0.67        25
          11       1.00      1.00      1.00        15
          12       1.00      0.48      0.65        21

    accuracy                           0.95      1150
   macro avg       0.91      0.86      0.87      1150
weighted avg       0.95      0.95      0.95      1150

